# DI 725 — Phase 2: Segmentation-Conditioned Prefix Tuning for BLIP

**Author:** Melih Can Hamurcu  
**Term Project:** Multi-modal Transformers — Remote Sensing Image Captioning

## What this notebook does

Phase 1 used a frozen BLIP captioning model with hard text prompts. The instructor flagged that
this is more of a "prompt engineering" exercise than a learned method, and pointed to **prompt
learning** as a legitimate research direction.

Phase 2 commits to that direction with a **bridge** twist: we train a small network that
maps the 7-dimensional segmentation class-percentage vector into a sequence of *learnable
prefix embeddings*, which are prepended to BLIP's text decoder input. BLIP itself stays frozen.

## Models compared

| Tag | Description | Trainable params |
|-----|-------------|------------------|
| **B0** | Zero-shot BLIP (Phase 1 baseline) | 0 |
| **B1** | BLIP + unconditional learned prefix (control: does prefix tuning alone help?) | ~6K |
| **M1** | BLIP + segmentation-conditioned prefix (this work) | ~800K |

## Evaluation

BLEU-1 / BLEU-4, METEOR, ROUGE-L on the held-out test split, against the
`hybrid_gemma3-4b` caption column as ground truth.


## 1. Environment

In [ ]:
!pip install -q \
    transformers==4.44.2 \
    accelerate==0.34.2 \
    evaluate==0.4.3 \
    nltk==3.9.1 \
    rouge-score==0.1.2 \
    wandb==0.18.5 \
    pillow pandas matplotlib tqdm

In [ ]:
import os, json, random, math, time
from dataclasses import dataclass, asdict
from typing import Optional, List, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration
import warnings
warnings.filterwarnings('ignore')

# NLTK data needed for METEOR
import nltk
for pkg in ['wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

All hyperparameters and paths in one place. Edit `CFG` to change a run.
`use_wandb=False` by default — set to True after `wandb login` to track runs.

In [ ]:
@dataclass
class Config:
    # paths
    base_dir: str = '/content/drive/MyDrive/DI725/DI725_project_dataset'
    output_dir: str = '/content/phase2_outputs'
    # model
    model_name: str = 'Salesforce/blip-image-captioning-base'
    # data
    caption_column: str = 'hybrid_gemma3-4b'
    train_frac: float = 0.80
    val_frac: float = 0.10  # rest is test
    seed: int = 42
    # training
    batch_size: int = 16
    eval_batch_size: int = 32
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    epochs: int = 3
    warmup_ratio: float = 0.05
    grad_clip: float = 1.0
    # bridge / prefix
    prefix_length: int = 8
    bridge_hidden: int = 256
    bridge_dropout: float = 0.1
    # tokenization / generation
    max_caption_length: int = 60
    max_gen_length: int = 50
    num_workers: int = 2
    # logging
    use_wandb: bool = False
    wandb_project: str = 'di725-phase2'
    wandb_run_name: str = ''  # set per model
    # eval — set small (e.g. 200) for a smoke test, None for full test set
    eval_subset: Optional[int] = None

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)

# reproducibility
torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)
random.seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

In [ ]:
# Optional: enable WandB for experiment tracking. Run `wandb login` first.
# CFG.use_wandb = True
if CFG.use_wandb:
    import wandb
    wandb.login()  # will prompt for API key if not already set

## 3. Data Loading and Splits

Phase 1 used 10/10 000 samples — flagged by the instructor. Phase 2 uses the full dataset
with a deterministic 80/10/10 train/val/test split (seed = 42). Indices are saved alongside
the run for full reproducibility.

In [ ]:
IMAGES_DIR = os.path.join(CFG.base_dir, 'images')
MASKS_DIR = os.path.join(CFG.base_dir, 'masks')
CAPTIONS_CSV = os.path.join(CFG.base_dir, 'captions.csv')

assert os.path.exists(IMAGES_DIR), f'missing: {IMAGES_DIR}'
assert os.path.exists(CAPTIONS_CSV), f'missing: {CAPTIONS_CSV}'

df = pd.read_csv(CAPTIONS_CSV)
print('rows:', len(df), '| columns:', df.columns.tolist())

# sanity: required columns are present
required = list(CFG.seg_columns) if hasattr(CFG, 'seg_columns') else ['Tree','Shrub','Grass','Crop','Built-up','Barren','Water']
for c in required + [CFG.caption_column, 'filename']:
    assert c in df.columns, f'missing column: {c}'

# Drop rows with missing caption or any seg column
df = df.dropna(subset=[CFG.caption_column] + required).reset_index(drop=True)
print('after dropna:', len(df))

df.head(3)

In [ ]:
# Deterministic split
rng = np.random.RandomState(CFG.seed)
perm = rng.permutation(len(df))
n_train = int(CFG.train_frac * len(df))
n_val = int(CFG.val_frac * len(df))
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f'train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}')

# Persist split indices for reproducibility
split_path = os.path.join(CFG.output_dir, 'splits.json')
with open(split_path, 'w') as f:
    json.dump({'train': train_idx.tolist(), 'val': val_idx.tolist(), 'test': test_idx.tolist(),
               'seed': CFG.seed, 'caption_column': CFG.caption_column}, f)
print('split indices saved to', split_path)

## 4. Dataset Class

The dataset returns the raw PIL image, the caption text, and the 7-dim segmentation vector
(scaled to [0, 1]). Tokenization and image preprocessing happen in the collate function so
the BLIP processor only loads once.

In [ ]:
SEG_COLS = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']

class RemoteSensingCaptionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, images_dir: str, caption_col: str):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.caption_col = caption_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        caption = str(row[self.caption_col]).strip()
        seg = np.array([row[c] for c in SEG_COLS], dtype=np.float32) / 100.0  # to [0,1]
        return {
            'image': image,
            'caption': caption,
            'seg': seg,
            'filename': row['filename'],
        }


def make_collate_fn(processor, max_length: int):
    def collate(batch):
        images = [b['image'] for b in batch]
        captions = [b['caption'] for b in batch]
        seg = torch.from_numpy(np.stack([b['seg'] for b in batch]))

        # Image features
        img_inputs = processor(images=images, return_tensors='pt')
        # Caption tokens (decoder input + labels)
        text_inputs = processor.tokenizer(
            captions,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        input_ids = text_inputs['input_ids']
        attention_mask = text_inputs['attention_mask']
        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        return {
            'pixel_values': img_inputs['pixel_values'],
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'seg': seg,
            'captions': captions,
            'filenames': [b['filename'] for b in batch],
        }
    return collate

## 5. Load Pretrained BLIP

We use the same checkpoint as Phase 1 (`Salesforce/blip-image-captioning-base`) so the B0
baseline is directly comparable.

In [ ]:
processor = BlipProcessor.from_pretrained(CFG.model_name)
blip_base = BlipForConditionalGeneration.from_pretrained(CFG.model_name).to(device)
blip_base.eval()
hidden_dim = blip_base.text_decoder.config.hidden_size
print('hidden_dim:', hidden_dim)

# Token IDs we need for generation
BOS_ID = processor.tokenizer.bos_token_id  # BLIP's [DEC]
EOS_ID = processor.tokenizer.sep_token_id  # BLIP uses [SEP] as end-of-sequence
PAD_ID = processor.tokenizer.pad_token_id
print('BOS:', BOS_ID, 'EOS:', EOS_ID, 'PAD:', PAD_ID)

In [ ]:
# Build dataloaders (we reuse one BlipProcessor for all models)
collate_fn = make_collate_fn(processor, CFG.max_caption_length)

train_ds = RemoteSensingCaptionDataset(train_df, IMAGES_DIR, CFG.caption_column)
val_ds = RemoteSensingCaptionDataset(val_df, IMAGES_DIR, CFG.caption_column)
test_ds = RemoteSensingCaptionDataset(test_df, IMAGES_DIR, CFG.caption_column)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG.eval_batch_size, shuffle=False,
                        num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG.eval_batch_size, shuffle=False,
                         num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=True)

print('batches: train=%d val=%d test=%d' % (len(train_loader), len(val_loader), len(test_loader)))

## 6. Model Architecture

### Shared scaffolding

Both **B1** (unconditional) and **M1** (seg-conditioned) inherit from `PrefixTunedBLIP`.
The base class freezes BLIP and implements:
- a custom `forward` that prepends the prefix to text-decoder inputs and masks the prefix
  region in the labels,
- a manual greedy `generate` (we can't use `model.generate()` because we're injecting
  embeddings rather than tokens).

Subclasses only differ in how they produce the prefix:
- **B1** holds a single `(K, D)` parameter tensor — same prefix for every example.
- **M1** has a small MLP that maps the 7-dim segmentation vector to `(K, D)` per example.


In [ ]:
class PrefixTunedBLIP(nn.Module):
    # BLIP with frozen weights + a learnable prefix prepended to the text decoder.

    def __init__(self, blip_model: BlipForConditionalGeneration, prefix_length: int = 8):
        super().__init__()
        self.blip = blip_model
        for p in self.blip.parameters():
            p.requires_grad = False
        self.blip.eval()  # keep frozen modules in eval mode (disables BLIP dropout)
        self.prefix_length = prefix_length
        self.hidden_dim = blip_model.text_decoder.config.hidden_size

    def get_prefix_embeds(self, seg: torch.Tensor, batch_size: int) -> torch.Tensor:
        # Return prefix tensor of shape (B, K, D). Override in subclasses.
        raise NotImplementedError

    def _encode_image(self, pixel_values: torch.Tensor):
        with torch.no_grad():
            vision_outputs = self.blip.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs[0]
        image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long,
                                device=image_embeds.device)
        return image_embeds, image_atts

    def forward(self,
                pixel_values: torch.Tensor,
                input_ids: torch.Tensor,
                attention_mask: torch.Tensor,
                labels: torch.Tensor,
                seg: Optional[torch.Tensor] = None):
        B = pixel_values.size(0)
        image_embeds, image_atts = self._encode_image(pixel_values)

        prefix_embeds = self.get_prefix_embeds(seg, B)  # (B, K, D)
        K = self.prefix_length

        word_embed = self.blip.text_decoder.bert.embeddings.word_embeddings
        text_embeds = word_embed(input_ids)  # (B, L, D)

        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
        prefix_mask = torch.ones(B, K, device=input_ids.device, dtype=attention_mask.dtype)
        full_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        # Mask prefix positions AND the [DEC] token in labels: we don't want the prefix
        # to be trained to predict the BOS token (it's a constant signal).
        labels_masked = labels.clone()
        labels_masked[:, 0] = -100  # the [DEC] / BOS position
        prefix_labels = torch.full((B, K), -100, dtype=labels.dtype, device=labels.device)
        full_labels = torch.cat([prefix_labels, labels_masked], dim=1)

        out = self.blip.text_decoder(
            inputs_embeds=inputs_embeds,
            attention_mask=full_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_atts,
            labels=full_labels,
            return_dict=True,
        )
        return out

    @torch.no_grad()
    def generate(self,
                 pixel_values: torch.Tensor,
                 seg: Optional[torch.Tensor] = None,
                 max_length: int = 50,
                 bos_id: int = BOS_ID,
                 eos_id: int = EOS_ID,
                 pad_id: int = PAD_ID) -> torch.Tensor:
        # Greedy autoregressive decode with the learned prefix as fixed context.
        self.eval()
        B = pixel_values.size(0)
        device = pixel_values.device
        image_embeds, image_atts = self._encode_image(pixel_values)
        prefix_embeds = self.get_prefix_embeds(seg, B)
        K = self.prefix_length

        word_embed = self.blip.text_decoder.bert.embeddings.word_embeddings
        input_ids = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_length):
            text_embeds = word_embed(input_ids)
            inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)
            full_mask = torch.ones(B, K + input_ids.size(1), dtype=torch.long, device=device)
            out = self.blip.text_decoder(
                inputs_embeds=inputs_embeds,
                attention_mask=full_mask,
                encoder_hidden_states=image_embeds,
                encoder_attention_mask=image_atts,
                return_dict=True,
            )
            next_logits = out.logits[:, -1, :]
            next_tok = next_logits.argmax(dim=-1, keepdim=True)
            # If a sequence is finished, pad it
            next_tok = torch.where(finished.unsqueeze(1),
                                   torch.full_like(next_tok, pad_id), next_tok)
            input_ids = torch.cat([input_ids, next_tok], dim=1)
            finished = finished | (next_tok.squeeze(1) == eos_id)
            if finished.all():
                break
        return input_ids

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

    def num_trainable(self):
        return sum(p.numel() for p in self.trainable_parameters())


class UnconditionalPrefixBLIP(PrefixTunedBLIP):
    # B1: a single learnable prefix shared across all inputs.
    def __init__(self, blip_model, prefix_length=8):
        super().__init__(blip_model, prefix_length)
        self.prefix = nn.Parameter(torch.randn(prefix_length, self.hidden_dim) * 0.02)

    def get_prefix_embeds(self, seg, batch_size):
        return self.prefix.unsqueeze(0).expand(batch_size, -1, -1)


class SegConditionedPrefixBLIP(PrefixTunedBLIP):
    # M1: prefix embeddings produced by a small MLP from the 7-dim seg vector.
    def __init__(self, blip_model, prefix_length=8, num_classes=7,
                 hidden=256, dropout=0.1):
        super().__init__(blip_model, prefix_length)
        out_dim = prefix_length * self.hidden_dim
        self.bridge = nn.Sequential(
            nn.Linear(num_classes, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )
        self.prefix_norm = nn.LayerNorm(self.hidden_dim)
        # Init last layer small so initial prefixes are near-zero
        nn.init.normal_(self.bridge[-1].weight, std=0.02)
        nn.init.zeros_(self.bridge[-1].bias)

    def get_prefix_embeds(self, seg, batch_size):
        assert seg is not None, 'M1 requires segmentation input'
        x = self.bridge(seg.to(next(self.parameters()).device))
        x = x.view(batch_size, self.prefix_length, self.hidden_dim)
        return self.prefix_norm(x)

In [ ]:
# Sanity check: build both models and count trainable params
m_b1 = UnconditionalPrefixBLIP(blip_base, prefix_length=CFG.prefix_length).to(device)
m_m1 = SegConditionedPrefixBLIP(blip_base, prefix_length=CFG.prefix_length,
                                hidden=CFG.bridge_hidden,
                                dropout=CFG.bridge_dropout).to(device)
print(f'B1 trainable: {m_b1.num_trainable():,}')
print(f'M1 trainable: {m_m1.num_trainable():,}')

# Forward smoke test on one batch
batch = next(iter(val_loader))
pv = batch['pixel_values'].to(device)
ii = batch['input_ids'].to(device)
am = batch['attention_mask'].to(device)
lb = batch['labels'].to(device)
sg = batch['seg'].to(device)
with torch.no_grad():
    out_b1 = m_b1(pv, ii, am, lb)
    out_m1 = m_m1(pv, ii, am, lb, sg)
print('B1 loss:', out_b1.loss.item(), '| M1 loss:', out_m1.loss.item())
del m_b1, m_m1, out_b1, out_m1; torch.cuda.empty_cache()

## 7. Training Loop

Standard AdamW + linear warmup-and-decay schedule, with gradient clipping and `torch.amp`
mixed precision. Only the prefix parameters (B1) or the bridge network (M1) receive
gradients — BLIP itself stays frozen end-to-end.

The training function is generic over the model class so we reuse it for both B1 and M1.

In [ ]:
def linear_warmup_cosine(step, total, warmup):
    if step < warmup:
        return step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def train_model(model, train_loader, val_loader, cfg, tag: str,
                seg_required: bool = False) -> dict:
    # Train a PrefixTunedBLIP model, return history dict.
    opt = torch.optim.AdamW(model.trainable_parameters(),
                            lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

    history = {'train_loss': [], 'val_loss': [], 'tag': tag}
    step = 0
    best_val = float('inf')
    ckpt_path = os.path.join(cfg.output_dir, f'{tag}_best.pt')

    if cfg.use_wandb:
        import wandb
        wandb.init(project=cfg.wandb_project, name=f'{tag}', config=asdict(cfg),
                   reinit=True)

    for epoch in range(cfg.epochs):
        model.train()
        # Keep BLIP submodules in eval mode (we froze them but .train() would re-enable dropout)
        model.blip.eval()
        epoch_losses = []
        pbar = tqdm(train_loader, desc=f'[{tag}] epoch {epoch+1}/{cfg.epochs}')
        for batch in pbar:
            pv = batch['pixel_values'].to(device, non_blocking=True)
            ii = batch['input_ids'].to(device, non_blocking=True)
            am = batch['attention_mask'].to(device, non_blocking=True)
            lb = batch['labels'].to(device, non_blocking=True)
            sg = batch['seg'].to(device, non_blocking=True) if seg_required else None

            # LR schedule
            lr_scale = linear_warmup_cosine(step, total_steps, warmup_steps)
            for g in opt.param_groups:
                g['lr'] = cfg.learning_rate * lr_scale

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                out = model(pv, ii, am, lb, sg) if seg_required else model(pv, ii, am, lb)
                loss = out.loss

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.trainable_parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()

            epoch_losses.append(loss.item())
            pbar.set_postfix(loss=f'{loss.item():.3f}', lr=f'{cfg.learning_rate*lr_scale:.2e}')
            if cfg.use_wandb:
                wandb.log({f'{tag}/train_loss': loss.item(),
                           f'{tag}/lr': cfg.learning_rate * lr_scale, 'step': step})
            step += 1

        train_loss = float(np.mean(epoch_losses))

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                pv = batch['pixel_values'].to(device, non_blocking=True)
                ii = batch['input_ids'].to(device, non_blocking=True)
                am = batch['attention_mask'].to(device, non_blocking=True)
                lb = batch['labels'].to(device, non_blocking=True)
                sg = batch['seg'].to(device, non_blocking=True) if seg_required else None
                with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                    out = model(pv, ii, am, lb, sg) if seg_required else model(pv, ii, am, lb)
                val_losses.append(out.loss.item())
        val_loss = float(np.mean(val_losses))

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'[{tag}] epoch {epoch+1}: train={train_loss:.4f}  val={val_loss:.4f}')
        if cfg.use_wandb:
            wandb.log({f'{tag}/epoch_train_loss': train_loss,
                       f'{tag}/epoch_val_loss': val_loss, 'epoch': epoch+1})

        # Save best
        if val_loss < best_val:
            best_val = val_loss
            # Save only trainable parts (much smaller than full BLIP)
            trainable_state = {k: v for k, v in model.state_dict().items()
                               if any(p is v for p in model.trainable_parameters())
                               or 'prefix' in k or 'bridge' in k}
            torch.save({'state_dict': trainable_state, 'config': asdict(cfg),
                        'epoch': epoch+1, 'val_loss': val_loss}, ckpt_path)
            print(f'  -> saved best to {ckpt_path}')

    history['best_val_loss'] = best_val
    history['ckpt_path'] = ckpt_path

    if cfg.use_wandb:
        wandb.finish()
    return history

### 7.1 Train B1 — unconditional learned prefix

In [ ]:
model_b1 = UnconditionalPrefixBLIP(blip_base, prefix_length=CFG.prefix_length).to(device)
print(f'B1 trainable params: {model_b1.num_trainable():,}')
hist_b1 = train_model(model_b1, train_loader, val_loader, CFG, tag='B1', seg_required=False)

### 7.2 Train M1 — segmentation-conditioned prefix

In [ ]:
model_m1 = SegConditionedPrefixBLIP(blip_base,
                                    prefix_length=CFG.prefix_length,
                                    hidden=CFG.bridge_hidden,
                                    dropout=CFG.bridge_dropout).to(device)
print(f'M1 trainable params: {model_m1.num_trainable():,}')
hist_m1 = train_model(model_m1, train_loader, val_loader, CFG, tag='M1', seg_required=True)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h, ax in zip([hist_b1, hist_m1], axes):
    ax.plot(range(1, len(h['train_loss'])+1), h['train_loss'], marker='o', label='train')
    ax.plot(range(1, len(h['val_loss'])+1),  h['val_loss'],  marker='s', label='val')
    ax.set_xlabel('epoch'); ax.set_ylabel('cross-entropy loss')
    ax.set_title(f"{h['tag']}  (best val = {h['best_val_loss']:.4f})")
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, 'training_curves.png'), dpi=200, bbox_inches='tight')
plt.show()

## 8. Evaluation on the Test Split

For each model we generate captions on the full test split (or `eval_subset` for a smoke test),
then compute BLEU-1, BLEU-4, METEOR, and ROUGE-L against the ground-truth captions.

In [ ]:
# Zero-shot BLIP (B0): use the model's built-in generate
@torch.no_grad()
def b0_generate(loader, max_subset=None):
    blip_base.eval()
    preds, refs, fns = [], [], []
    n = 0
    for batch in tqdm(loader, desc='B0 zero-shot generate'):
        pv = batch['pixel_values'].to(device)
        out_ids = blip_base.generate(pixel_values=pv, max_new_tokens=CFG.max_gen_length,
                                     num_beams=1)
        for i in range(out_ids.size(0)):
            cap = processor.decode(out_ids[i], skip_special_tokens=True).strip()
            preds.append(cap); refs.append(batch['captions'][i]); fns.append(batch['filenames'][i])
            n += 1
            if max_subset is not None and n >= max_subset:
                return preds, refs, fns
    return preds, refs, fns


@torch.no_grad()
def prefix_generate(model, loader, seg_required=False, max_subset=None):
    model.eval()
    preds, refs, fns = [], [], []
    n = 0
    for batch in tqdm(loader, desc=f'generate'):
        pv = batch['pixel_values'].to(device)
        sg = batch['seg'].to(device) if seg_required else None
        out_ids = model.generate(pv, seg=sg, max_length=CFG.max_gen_length,
                                 bos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID)
        for i in range(out_ids.size(0)):
            cap = processor.decode(out_ids[i], skip_special_tokens=True).strip()
            preds.append(cap); refs.append(batch['captions'][i]); fns.append(batch['filenames'][i])
            n += 1
            if max_subset is not None and n >= max_subset:
                return preds, refs, fns
    return preds, refs, fns

In [ ]:
# Generate for all three models
sub = CFG.eval_subset
print('=== B0 (zero-shot) ===')
preds_b0, refs, fns = b0_generate(test_loader, max_subset=sub)

print('=== B1 (unconditional prefix) ===')
preds_b1, _, _ = prefix_generate(model_b1, test_loader, seg_required=False, max_subset=sub)

print('=== M1 (seg-conditioned prefix) ===')
preds_m1, _, _ = prefix_generate(model_m1, test_loader, seg_required=True, max_subset=sub)

# Save raw predictions for the report
gen_df = pd.DataFrame({
    'filename': fns,
    'reference': refs,
    'B0_zero_shot': preds_b0,
    'B1_uncond_prefix': preds_b1,
    'M1_seg_prefix': preds_m1,
})
pred_path = os.path.join(CFG.output_dir, 'test_predictions.csv')
gen_df.to_csv(pred_path, index=False)
print('saved predictions to', pred_path)
gen_df.head(5)

In [ ]:
# Metrics: BLEU-1, BLEU-4, METEOR, ROUGE-L
import evaluate
bleu_metric = evaluate.load('bleu')
meteor_metric = evaluate.load('meteor')
rouge_metric = evaluate.load('rouge')


def compute_metrics(preds, refs):
    # bleu wants list-of-references-per-prediction
    refs_nested = [[r] for r in refs]
    bleu1 = bleu_metric.compute(predictions=preds, references=refs_nested, max_order=1)['bleu']
    bleu4 = bleu_metric.compute(predictions=preds, references=refs_nested, max_order=4)['bleu']
    meteor = meteor_metric.compute(predictions=preds, references=refs)['meteor']
    rouge = rouge_metric.compute(predictions=preds, references=refs)['rougeL']
    return {'BLEU-1': bleu1, 'BLEU-4': bleu4, 'METEOR': meteor, 'ROUGE-L': rouge}


results = {
    'B0_zero_shot': compute_metrics(preds_b0, refs),
    'B1_uncond_prefix': compute_metrics(preds_b1, refs),
    'M1_seg_prefix': compute_metrics(preds_m1, refs),
}
results_df = pd.DataFrame(results).T
results_df.to_csv(os.path.join(CFG.output_dir, 'results_table.csv'))
print(results_df.round(4))

In [ ]:
# Bar chart of the four metrics across the three models
fig, ax = plt.subplots(figsize=(9, 5))
metrics_order = ['BLEU-1', 'BLEU-4', 'METEOR', 'ROUGE-L']
x = np.arange(len(metrics_order))
width = 0.27
for i, model_name in enumerate(['B0_zero_shot', 'B1_uncond_prefix', 'M1_seg_prefix']):
    vals = [results[model_name][m] for m in metrics_order]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=model_name)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics_order)
ax.set_ylabel('score'); ax.set_title('Captioning metrics on test split')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, 'metrics_bar.png'), dpi=200, bbox_inches='tight')
plt.show()

## 9. Qualitative Comparison

Side-by-side captions for a handful of test images. The B0 baseline often hallucinates
out-of-domain objects (planes, birds); B1 reduces this through prefix tuning; M1 grounds
the description in the actual scene composition via the segmentation signal.

In [ ]:
def show_qualitative(idx_list):
    n = len(idx_list)
    fig, axes = plt.subplots(n, 1, figsize=(7, 6 * n))
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, idx_list):
        row = gen_df.iloc[idx]
        img = Image.open(os.path.join(IMAGES_DIR, row['filename'])).convert('RGB')
        ax.imshow(img); ax.axis('off')
        ax.set_title(row['filename'])
        text = (f"GT : {row['reference']}\n"
                f"B0 : {row['B0_zero_shot']}\n"
                f"B1 : {row['B1_uncond_prefix']}\n"
                f"M1 : {row['M1_seg_prefix']}")
        ax.text(1.02, 0.5, text, transform=ax.transAxes, va='center', ha='left',
                wrap=True, fontsize=9, family='monospace')
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, 'qualitative.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

# Pick 4 diverse examples
rng2 = np.random.RandomState(0)
qual_idx = rng2.choice(len(gen_df), size=min(4, len(gen_df)), replace=False).tolist()
show_qualitative(qual_idx)

## 10. Run Summary

Everything needed to reproduce this run is now in `CFG.output_dir`:
splits, predictions, metrics, training curves, qualitative figure.

In [ ]:
summary = {
    'config': asdict(CFG),
    'B1_history': hist_b1,
    'M1_history': hist_m1,
    'results': results,
    'n_train': len(train_ds), 'n_val': len(val_ds), 'n_test': len(test_ds),
}
summary_path = os.path.join(CFG.output_dir, 'run_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('summary saved to', summary_path)

print('\n=== Final results ===')
print(results_df.round(4))
print('\nFiles in', CFG.output_dir, ':')
for f in sorted(os.listdir(CFG.output_dir)):
    print(' ', f)